# 09 — Quantized Multi-Hop Reasoning on Tiny Aya (3.35B)
## قياس أثر التكميم والاستدلال متعدد الخطوات في النماذج متعددة اللغات أصلاً (Tiny Aya Earth)

> **The Empirical Puzzle & Research Hypothesis:**
> In notebooks `07` and `08`, we evaluated Google's `gemma-2-2b` on 2-hop factual reasoning across English, Modern Standard Arabic (MSA), and Egyptian Arabic (Masri). We uncovered two critical vulnerabilities:
> 1. **The L22 English Pivot:** Despite receiving Arabic prompts, Gemma-2 resolves multi-hop factual relations ($[\text{Suez Canal}] \to [\text{Egypt}] \to [\text{Cairo}]$) by routing through an internal English-dominated subspace at Layer 22 ($P(\text{' Cairo'}) = 0.975$) before attempting a late cross-lingual un-embedding projection.
> 2. **Asymmetric Quantization Penalty:** Low-bit weight quantization (Int4 RTN) shattered this late cross-lingual projection, causing a $-65.8\%$ factual drop in MSA and a $-72.9\%$ drop in Masri, with Masri collapsing into a conversational question-mark attractor ($P('?') = 0.994$).
>
> **The Tokenization Nuance & The Hybrid Evaluation Methodology:**
> In Arabic language modeling, the definite article `'ال'` is a high-frequency unigram prefix. Without contextual whitespace, Arabic entity names decompose into subwords (e.g. `['ال', 'قاهرة']`, token IDs `[379, 246588]`). Tracking `'ال'` alone risks measuring generic grammatical prefixing rather than true factual retrieval.
>
> To resolve this with empirical rigor, we implement a **Hybrid Evaluation Framework**:
> 1. **Unified Single-Token Logit Lens:** By structuring the assistant turn to terminate in a natural whitespace-delimited connector (`"The capital is "` / `"العاصمة هي "` / `"عاصمة البلد هي "`), the model naturally predicts the **unified single-token** entity name: `' Cairo'` (ID `147633`) in English, and `' القاهرة'` (ID `97936`) in Arabic. This preserves clean 1-token logit lens tracking across all 36 transformer layers.
> 2. **Candidate City Contrastive Discrimination:** At the final un-embedding layer, we contrast $P(\text{Cairo})$ against a battery of competing Egyptian cities (`Alexandria`, `Giza`, `Suez`, `Aswan`, `Mansoura`, `Tanta`). Demonstrating a high contrast ratio ($P(\text{Cairo}) / P(\text{Alexandria}) \gg 1$) proves genuine factual knowledge retrieval over generic linguistic probability.
>
> **Core Questions for Tiny Aya Earth (3.35B):**
> - **H1 (Direct Arabic Circuit):** Does native multilingual pre-training bypass the English lingua franca pivot, resolving Arabic facts directly within an Arabic semantic subspace?
> - **H2 (Dialectal Quantization Resilience):** Does native multilingual representation confer structural resilience against low-bit quantization (Int8 and Int4), preventing the dialectal collapse seen in Gemma-2-2B?
> - **H3 (RTN vs Distribution-Aware NF4):** Does information-theoretically optimal 4-bit quantization (NormalFloat4) protect fragile dialectal representations where naive Round-To-Nearest (RTN) fails?

### Step 1: Environment Configuration & ROCm Execution Sanity Check

We initialize the execution environment on AMD Strix Halo using ROCm. We configure PyTorch to use `HSA_OVERRIDE_GFX_VERSION=11.0.0` to target the RDNA 3.5 iGPU architecture cleanly, disable tokenizers parallelism to eliminate fork warnings, and assert CUDA/ROCm device availability.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")


PyTorch Version: 2.5.1+rocm6.2
Device: cuda
Device Name: AMD Radeon Graphics
Allocated Memory: 0.00 MB


amdgpu.ids: No such file or directory


### Step 2: Parallel Multi-Hop Queries, Prefill Framing & Tokenizer Diagnostics

To enable rigorous logit lens evaluation on Tiny Aya without definite-article ambiguity, we format our 2-hop reasoning queries using Aya's native chat template with an assistant response prefill ending in a trailing space:
- **English:** `... <|START_RESPONSE|>The capital is `
- **MSA:** `... <|START_RESPONSE|>العاصمة هي `
- **Masri:** `... <|START_RESPONSE|>عاصمة البلد هي `

**Tokenizer Diagnostic Insight:**
Tiny Aya uses a 262,144-vocabulary `Cohere2` BPE tokenizer. Because BPE treats leading whitespace as part of the token:
- Leading space + Cairo (`' القاهرة'`) maps to a **single unified token ID `97936`**.
- Leading space + Cairo in English (`' Cairo'`) maps to a **single unified token ID `147633`**.
- Competing cities decompose into distinct initial tokens (`' الإ'` for Alexandria, `' الس'` for Suez, `' الجيزة'` for Giza).

Below, we construct the prompts and inspect the tokenization diagnostics.

In [2]:
TOKENIZER_NAME = "CohereLabs/tiny-aya-earth"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

RAW_QUERIES = {
    "EN": ("What is the capital of the country where the Suez Canal is located?", "The capital is "),
    "MSA": ("ما هي عاصمة الدولة التي تقع فيها قناة السويس؟", "العاصمة هي "),
    "Masri": ("ايه هي عاصمة البلد اللي فيها قناة السويس؟", "عاصمة البلد هي "),
}

PROMPTS = {}
print("=== Formatted Multi-Hop Chat Prompts with Assistant Prefill ===")
for lang, (q, prefill) in RAW_QUERIES.items():
    msgs = [{"role": "user", "content": q}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prefill
    PROMPTS[lang] = formatted
    tok_ids = tokenizer.encode(formatted, add_special_tokens=False)
    print(f"\n[{lang:<5}] Total Prompt Length: {len(tok_ids)} tokens")
    print(f"Tail: ...{formatted[-60:]!r}")

# Diagnostic verification of target tokens and candidate cities
CANDIDATE_CITIES = {
    "Cairo": " القاهرة",
    "Alexandria": " الإسكندرية",
    "Giza": " الجيزة",
    "Suez": " السويس",
    "Aswan": " أسوان",
    "Mansoura": " المنصورة",
    "Tanta": " طنطا",
}

print("\n=== Tokenizer Diagnostics: Candidate Egyptian Cities ===")
for name, word in CANDIDATE_CITIES.items():
    ids = tokenizer.encode(word, add_special_tokens=False)
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"{name:12} | Word: {repr(word):14} | IDs: {str(ids):25} | Decoded: {decoded}")


=== Formatted Multi-Hop Chat Prompts with Assistant Prefill ===

[EN   ] Total Prompt Length: 383 tokens
Tail: ...'RN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>The capital is '

[MSA  ] Total Prompt Length: 381 tokens
Tail: ...'F_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>العاصمة هي '

[Masri] Total Prompt Length: 381 tokens
Tail: ...'RN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>عاصمة البلد هي '

=== Tokenizer Diagnostics: Candidate Egyptian Cities ===
Cairo        | Word: ' القاهرة'     | IDs: [97936]                   | Decoded: [' القاهرة']
Alexandria   | Word: ' الإسكندرية'  | IDs: [4868, 45797, 2960, 14267] | Decoded: [' الإ', 'سك', 'ند', 'رية']
Giza         | Word: ' الجيزة'      | IDs: [153046]                  | Decoded: [' الجيزة']
Suez         | Word: ' السويس'      | IDs: [2658, 78092]             | Decoded: [' الس', 'ويس']
Aswan        | Word: ' أسوان'       | IDs: [11637, 6291]             | Decoded: [' أس', 'وان']
Mansoura     | Word: ' المنصورة'    | IDs: [9525, 7831

### Step 3: Pure PyTorch Weight-Only Quantization Engine (Int8, Int4 RTN, NF4)

To compare directly with Gemma-2-2B without platform-specific CUDA dependencies (like `bitsandbytes`), we implement a pure PyTorch weight-only quantization engine compatible with ROCm:
1. **Int8 (Symmetric Per-Tensor):** Standard 8-bit integer quantization mapping weights to $[-128, 127]$.
2. **Int4 RTN (Asymmetric Per-Group Round-To-Nearest):** 16 group bins with scale and zero-point ($block\_size = 128$).
3. **NF4 (NormalFloat4):** Quantile-optimal 4-bit representation for normally distributed neural weights ($block\_size = 64$), matching QLoRA's information-theoretic design.

A model wrapper dynamically swaps linear projection weights across precisions and restores unquantized FP16 weights without reloading from disk.

In [3]:
# Pure PyTorch Weight-Only Quantization Engine (No CUDA-only dependencies)

NF4_QUANT_TABLE = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
], device=DEVICE, dtype=torch.float16)

def quantize_weight_int8(w: torch.Tensor) -> torch.Tensor:
    scale = w.abs().max() / 127.0
    w_q = torch.clamp(torch.round(w / scale), -128, 127).to(torch.int8)
    return (w_q.to(w.dtype) * scale).contiguous()

def quantize_weight_int4_rtn(w: torch.Tensor, block_size: int = 128) -> torch.Tensor:
    orig_shape = w.shape
    w_flat = w.reshape(-1, block_size)
    mn = w_flat.min(dim=-1, keepdim=True).values
    mx = w_flat.max(dim=-1, keepdim=True).values
    scale = torch.clamp((mx - mn) / 15.0, min=1e-8)
    q = torch.clamp(torch.round((w_flat - mn) / scale), 0, 15)
    return (q * scale + mn).reshape(orig_shape).contiguous()

def quantize_weight_nf4(w: torch.Tensor, block_size: int = 64) -> torch.Tensor:
    orig_shape = w.shape
    w_flat = w.reshape(-1, block_size)
    abs_max = torch.clamp(w_flat.abs().max(dim=-1, keepdim=True).values, min=1e-8)
    norm_w = w_flat / abs_max
    dists = torch.abs(norm_w.unsqueeze(-1) - NF4_QUANT_TABLE)
    best_idx = torch.argmin(dists, dim=-1)
    dequant = NF4_QUANT_TABLE[best_idx]
    return (dequant * abs_max).reshape(orig_shape).contiguous()

class ModelQuantizer:
    def __init__(self, model: torch.nn.Module):
        self.model = model
        self.original_weights = {}
        for name, module in self.model.named_modules():
            if isinstance(module, torch.nn.Linear) and "lm_head" not in name:
                self.original_weights[name] = module.weight.detach().clone()
        print(f"Quantizer initialized: cached {len(self.original_weights)} linear layers.")

    def apply_quantization(self, mode: str):
        if mode == "FP16":
            for name, orig_w in self.original_weights.items():
                dict(self.model.named_modules())[name].weight.data.copy_(orig_w)
            return

        for name, orig_w in self.original_weights.items():
            mod = dict(self.model.named_modules())[name]
            if mode == "Int8":
                mod.weight.data.copy_(quantize_weight_int8(orig_w))
            elif mode == "Int4 RTN":
                mod.weight.data.copy_(quantize_weight_int4_rtn(orig_w))
            elif mode == "NF4":
                mod.weight.data.copy_(quantize_weight_nf4(orig_w))
            else:
                raise ValueError(f"Unknown mode: {mode}")

    def restore_fp16(self):
        self.apply_quantization("FP16")


/tmp/ipykernel_259373/1944891803.py:3: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  NF4_QUANT_TABLE = torch.tensor([


### Step 4: Loading Tiny Aya Earth & Target Token Resolution

We load `CohereLabs/tiny-aya-earth` (3.35B, 36 layers) onto our ROCm accelerator in native `bfloat16`/`float16`. 

We register the target tokens corresponding to the single-token completions induced by our assistant prefill:
- **English Target:** `' Cairo'` (Token ID: `147633`)
- **MSA Target:** `' القاهرة'` (Token ID: `97936`)
- **Masri Target:** `' القاهرة'` (Token ID: `97936`)

We verify token resolution and initialize the `ModelQuantizer`.

In [4]:
MODEL_NAME = "CohereLabs/tiny-aya-earth"
print(f"Loading {MODEL_NAME} onto {DEVICE} in float16...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=DEVICE,
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Loaded successfully: {NUM_LAYERS} transformer layers.")
print(f"Logit scale: {getattr(model, 'logit_scale', 1.0)}")

TARGET_TOKENS = {
    "EN": " Cairo",
    "MSA": " القاهرة",
    "Masri": " القاهرة",
}

TARGET_IDS = {
    lang: tokenizer.encode(tok_str, add_special_tokens=False)[0]
    for lang, tok_str in TARGET_TOKENS.items()
}

print("\n=== Registered Target Tokens ===")
for lang, tid in TARGET_IDS.items():
    print(f"[{lang:<5}] Target Token ID: {tid:6d} | Decoded: {repr(tokenizer.decode([tid]))}")

quantizer = ModelQuantizer(model)


Loading CohereLabs/tiny-aya-earth onto cuda in float16...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded successfully: 36 transformer layers.
Logit scale: 1.0

=== Registered Target Tokens ===
[EN   ] Target Token ID: 147633 | Decoded: ' Cairo'
[MSA  ] Target Token ID:  97936 | Decoded: ' القاهرة'
[Masri] Target Token ID:  97936 | Decoded: ' القاهرة'
Quantizer initialized: cached 252 linear layers.


### Step 5: Multi-Layer Logit Lens Trajectory Across Precisions

We evaluate the prompt triples across **four precision regimes**:
1. **FP16 Baseline**
2. **Int8 Symmetric**
3. **Int4 RTN (Round-to-Nearest)**
4. **NF4 (NormalFloat4)**

For each precision and language, we extract the hidden states at all 36 layers at the final prefill sequence position.
We apply the final RMSNorm (`model.model.norm`) and project through the un-embedding matrix (`model.lm_head`) scaled by `model.logit_scale`. We compute the softmax probability assigned to the target token (`' Cairo'` or `' القاهرة'`) across layers.

In [5]:
records = []

def evaluate_precision(precision_label: str):
    print(f"\nEvaluating regime: {precision_label}...")
    quantizer.apply_quantization(precision_label)
    
    with torch.no_grad():
        for lang, prompt_text in PROMPTS.items():
            target_id = TARGET_IDS[lang]
            inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
            outputs = model(**inputs, output_hidden_states=True)
            
            for layer_idx in range(NUM_LAYERS + 1):
                h = outputs.hidden_states[layer_idx][:, -1, :] # [1, hidden_dim]
                
                if layer_idx < NUM_LAYERS:
                    h_norm = model.model.norm(h)
                else:
                    h_norm = h
                    
                logits = model.lm_head(h_norm) * getattr(model, "logit_scale", 1.0)
                probs = torch.softmax(logits, dim=-1)
                target_p = probs[0, target_id].item()
                
                records.append({
                    "model": "tiny-aya-earth-3.35b",
                    "precision": precision_label,
                    "language": lang,
                    "layer": layer_idx,
                    "target_prob": target_p,
                    "target_token": TARGET_TOKENS[lang],
                    "target_id": target_id
                })

for prec in ["FP16", "Int8", "Int4 RTN", "NF4"]:
    evaluate_precision(prec)

quantizer.restore_fp16()

df_results = pd.DataFrame(records)
df_results.to_csv("quantization_tiny_aya_results.csv", index=False)
print("\nLogit lens evaluation complete. Saved results to quantization_tiny_aya_results.csv")

final_layer_summary = df_results[df_results["layer"] == NUM_LAYERS].pivot(
    index="precision", columns="language", values="target_prob"
)[["EN", "MSA", "Masri"]]
print("\n=== Final Layer Output Probabilities P(Cairo) ===")
print((final_layer_summary * 100).round(2).astype(str) + "%")



Evaluating regime: FP16...


/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformers/models/cohere2/modeling_cohere2.py:71: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash attention support on Navi31 GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at ../aten/src/ATen/native/transformers/hip/sdp_utils.cpp:225.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Memory Efficient attention on Navi31 GPU is still experimental. Enable it wi


Evaluating regime: Int8...



Evaluating regime: Int4 RTN...



Evaluating regime: NF4...



Logit lens evaluation complete. Saved results to quantization_tiny_aya_results.csv

=== Final Layer Output Probabilities P(Cairo) ===
language       EN     MSA   Masri
precision                        
FP16       50.05%  32.28%  21.85%
Int4 RTN   26.95%  13.77%  11.46%
Int8        64.4%  31.13%  21.25%
NF4        42.36%  22.84%  12.85%
